# 1. Setup & Data Loading

## 1.1 Import Libraries
We import the essential libraries for data manipulation, visualization, and analysis.

In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Display settings
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Setup complete.")

Setup complete.


## 1.2 Load Dataset
We load the CSV provided by Alkemy. This dataset contains 3,248 tasks from real business operations, with 34 variables covering the full task lifecycle: input, process, output, and value.

In [84]:
df = pd.read_csv('ai_productivity_dataset_final.csv')
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Shape: 3248 rows × 34 columns


,task_id,client,project_id,client_tier,team,task_type,seniority,task_complexity_score,brief_quality_score,deadline_pressure,scope_change_flag,pricing_model,created_at,delivered_at,sla_days,sla_breach,hours_spent,billable_hours,ai_usage_pct,ai_assisted,revisions,errors,rework_hours,outcome_score,revenue,cost,profit,created_by,updated_at,task_status,workflow_stage,jira_ticket,legacy_ai_flag,content_version
0,T00000,Client_F,P038,mid,Content,report,junior,2,3.00,high,0,hourly,2025-11-20,2025-11-25,10.00,0,7.63,5.14,0.75,True,1,1,2.10,69.93,498.11,346.17,151.94,user_096,2025-11-28,review,finalized,JIRA-49014,true,v1
1,T00001,Client_H,P028,low,Paid Media,release,junior,1,2.00,medium,0,fixed,2026-01-24,2026-01-26,7.00,0,9.52,8.22,0.12,False,1,1,4.48,82.79,847.01,343.18,503.83,user_058,2026-01-26,delivered,client_review,JIRA-84793,false,v1
2,T00002,Client_D,P009,low,Design,dev,junior,3,4.00,medium,0,fixed,2025-09-16,2025-09-23,5.00,1,8.45,6.15,0.37,True,2,0,2.71,76.40,1374.07,365.02,1009.05,user_074,2025-09-17,in_progress,qa,JIRA-42485,true,v2
3,T00003,Client_E,P023,mid,Content,design,mid,3,2.00,low,0,hourly,2025-11-06,2025-11-09,3.00,0,28.35,24.22,0.07,False,4,1,0.00,NaN,2379.11,1514.73,864.38,user_011,2025-11-12,in_progress,briefing,JIRA-53111,false,v1
4,T00004,Client_C,P014,low,Design,article,senior,2,5.00,low,0,fixed,2026-05-02,2026-05-06,7.00,0,5.93,4.44,0.20,True,2,2,0.84,NaN,709.95,335.27,374.68,user_007,2026-05-09,review,execution,JIRA-86006,true,v2


## 1.3 Column-by-Column Audit
Before any cleaning, we need to understand what each column contains: data type, missing values, unique values, and potential issues. We split the 34 columns into logical groups for inspection.

In [85]:
# Full schema overview
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notnull().sum(),
    'null': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'nunique': df.nunique(),
    'sample_values': [df[col].dropna().unique()[:5].tolist() for col in df.columns]
})
audit

,dtype,non_null,null,null_pct,nunique,sample_values
task_id,object,3248,0,0.00,3200,"[T00000, T00001, T00002, T00003, T00004]"
client,object,3248,0,0.00,28,"[Client_F, Client_H, Client_D, Client_E, Clien..."
project_id,object,3248,0,0.00,64,"[P038, P028, P009, P023, P014]"
client_tier,object,3248,0,0.00,3,"[mid, low, high]"
team,object,3248,0,0.00,15,"[Content, Paid Media, Design, seo, Media]"
task_type,object,3248,0,0.00,29,"[report, release, dev, design, article]"
seniority,object,3248,0,0.00,3,"[junior, mid, senior]"
task_complexity_score,int64,3248,0,0.00,5,"[2, 1, 3, 4, 5]"
brief_quality_score,float64,3179,69,2.12,5,"[3.0, 2.0, 4.0, 5.0, 1.0]"
deadline_pressure,object,3248,0,0.00,3,"[high, medium, low]"


commentare tabella

# 2. Data Cleaning

## 1. task_id

`task_id` is the unique identifier for each task. 

In [86]:
print(f"Null values: {df['task_id'].isnull().sum()}")
print(f"Unique values: {df['task_id'].nunique()} (expected: {len(df)})")
print(f"Duplicates: {df['task_id'].duplicated().sum()}")
print(f"\nFormat sample: {df['task_id'].head(5).tolist()}")
print(f"Pattern check - all start with 'T': {df['task_id'].str.startswith('T').all()}")

Null values: 0
Unique values: 3200 (expected: 3248)
Duplicates: 48

Format sample: ['T00000', 'T00001', 'T00002', 'T00003', 'T00004']
Pattern check - all start with 'T': True


In [87]:
# Find which task_ids are duplicated
dup_ids = df[df['task_id'].duplicated(keep=False)].sort_values('task_id')
print(f"Total rows involved in duplicates: {len(dup_ids)}")
print(f"Unique duplicated IDs: {dup_ids['task_id'].nunique()}")

# Show a few examples — are the rows identical or different?
print("\nExample duplicated rows:")
sample_dup_id = dup_ids['task_id'].iloc[0]
print(f"\nRows with task_id = '{sample_dup_id}':")
print(df[df['task_id'] == sample_dup_id].to_string())

Total rows involved in duplicates: 96
Unique duplicated IDs: 48

Example duplicated rows:

Rows with task_id = 'T00010':
     task_id    client project_id client_tier   team task_type seniority  task_complexity_score  brief_quality_score deadline_pressure  scope_change_flag pricing_model  created_at delivered_at  sla_days  sla_breach  hours_spent  billable_hours  ai_usage_pct  ai_assisted  revisions  errors  rework_hours  outcome_score  revenue   cost  profit created_by  updated_at  task_status workflow_stage jira_ticket legacy_ai_flag content_version
10    T00010  Client_F       P020         low  Media        ad       mid                      4                 3.00               low                  0         fixed  2025-09-11   2025-09-14      2.00           1        11.57            7.00          0.38         True          5       1          4.59          51.37   743.55 756.78  -13.23   user_064  2025-09-12    delivered  client_review  JIRA-82158          false           final
3233 

In [88]:
# Show all 48 duplicate pairs — compare key differing fields
dup_ids_list = df[df['task_id'].duplicated(keep=False)]['task_id'].unique()

comparison_cols = ['task_id', 'client', 'created_by', 'updated_at', 'task_status', 
                   'jira_ticket', 'legacy_ai_flag', 'content_version']

for tid in dup_ids_list[:49]:  # first 10 pairs
    pair = df[df['task_id'] == tid][comparison_cols]
    print(pair.to_string())
    print("-" * 80)

print(f"\n... showing the {len(dup_ids_list)} duplicate pairs")

     task_id    client created_by  updated_at  task_status jira_ticket legacy_ai_flag content_version
10    T00010  Client_F   user_064  2025-09-12    delivered  JIRA-82158          false           final
3233  T00010  Client_J   user_078  2025-09-11  in_progress  JIRA-63281        unknown              v3
--------------------------------------------------------------------------------
     task_id    client created_by  updated_at  task_status jira_ticket legacy_ai_flag content_version
139   T00139  Client_H   user_011  2026-03-04  in_progress  JIRA-21055           true              v1
3202  T00139  Client_H   user_059  2026-03-05    delivered  JIRA-92895          false              v4
--------------------------------------------------------------------------------
     task_id    client created_by  updated_at  task_status jira_ticket legacy_ai_flag content_version
170   T00170  Client_H   user_003  2025-09-10  in_progress  JIRA-59955           true              v3
3237  T00170  Client_H

**Finding:** All 48 duplicate pairs share identical core data (hours, financials, scores, AI usage) but differ in metadata fields (`client`, `created_by`, `updated_at`, `task_status`, `jira_ticket`, `legacy_ai_flag`, `content_version`). This is consistent with system-level duplicates — the same task recorded twice with different metadata states.

Some pairs also differ in `client` (e.g. T00010: Client_F vs Client_J), suggesting possible data entry errors in the duplicate row.

**Decision:** For each pair, we keep the row with the **most recent `updated_at`**, as it represents the latest known state of the task. This drops 48 rows (1.5% of data) with no loss of analytical variables.

In [89]:
rows_before = len(df)

# Sort by updated_at descending so the most recent comes first
df = df.sort_values('updated_at', ascending=False)

# Keep first occurrence (most recent) for each task_id
df = df.drop_duplicates(subset='task_id', keep='first')

# Restore original order
df = df.sort_values('task_id').reset_index(drop=True)

rows_after = len(df)
print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Dropped:     {rows_before - rows_after} duplicate rows")
print(f"Unique task_ids: {df['task_id'].nunique()} (expected: {rows_after})")

Rows before: 3248
Rows after:  3200
Dropped:     48 duplicate rows
Unique task_ids: 3200 (expected: 3200)


## 2. client

`client` identifies which client the task belongs to.

In [90]:
print(f"Null values: {df['client'].isnull().sum()}")
print(f"Unique values: {df['client'].nunique()}")
print(f"\nDistribution:")
print(df['client'].value_counts())

Null values: 0
Unique values: 28

Distribution:
client
Client_G    410
Client_B    392
Client_E    389
Client_H    385
Client_D    381
Client_C    371
Client_A    358
Client_F    356
Client_L     13
Client_Z     13
Client_[     11
Client_K     11
Client_S     10
Client_U     10
Client_T     10
Client_R      9
Client_\      9
Client_Q      8
Client_N      7
Client_Y      7
Client_M      7
Client_P      6
Client_W      5
Client_J      5
Client_O      5
Client_X      5
Client_I      4
Client_V      3
Name: count, dtype: int64


**Finding:** 28 unique clients, but a clear split:
- **8 main clients** (A through H): 356-410 records each, totaling ~97% of the dataset
- **20 minor clients** (I through Z plus `Client_[` and `Client_\`): 3-13 records each

`Client_[` (11 records) and `Client_\` (9 records) appear to be encoding/data entry errors — brackets and backslashes are not valid client names.

**Decision:** No cleaning needed for the main 8 clients. For the minor clients, we flag `Client_[` and `Client_\` as likely data errors but keep all records — we don't have enough context to reassign them, and dropping 20 records would be arbitrary. We will note this as a data quality issue.

For downstream analysis, we may want to group minor clients as "Other" to avoid noise from small sample sizes.

In [91]:
# Flag suspicious client names
suspicious = df[df['client'].isin(['Client_[', 'Client_\\'])]
print(f"Suspicious client names: {len(suspicious)} records")
print(suspicious[['task_id', 'client', 'project_id', 'client_tier']].to_string())

Suspicious client names: 20 records
     task_id    client project_id client_tier
455   T00455  Client_[       P017         mid
494   T00494  Client_[       P009         mid
543   T00543  Client_[       P018        high
664   T00664  Client_\       P043         mid
771   T00771  Client_[       P041         mid
984   T00984  Client_[       P061        high
1455  T01455  Client_\       P025         low
1460  T01460  Client_\       P031         mid
1529  T01529  Client_[       P004         low
1610  T01610  Client_[       P038         mid
1613  T01613  Client_\       P048         mid
1848  T01848  Client_[       P010         low
1980  T01980  Client_[       P052         mid
2069  T02069  Client_\       P034        high
2423  T02423  Client_[       P051         low
2552  T02552  Client_\       P041        high
2661  T02661  Client_\       P020         mid
2743  T02743  Client_[       P041         low
2775  T02775  Client_\       P039         low
2933  T02933  Client_\       P035        hig

**Result:** The 20 suspicious records (`Client_[` and `Client_\`) are scattered across different projects and tiers with no clear pattern for reassignment. We keep them as-is and flag this as a known data quality issue. These likely result from encoding errors in the source system.


## 3. project_id

`project_id` links tasks to projects.

In [92]:
print(f"Null values: {df['project_id'].isnull().sum()}")
print(f"Unique values: {df['project_id'].nunique()}")
print(f"Format sample: {df['project_id'].head(5).tolist()}")
print(f"All start with 'P': {df['project_id'].str.startswith('P').all()}")
print(f"\nTasks per project:")
print(df['project_id'].value_counts().describe())

Null values: 0
Unique values: 64
Format sample: ['P038', 'P028', 'P009', 'P023', 'P014']
All start with 'P': True

Tasks per project:
count   64.00
mean    50.00
std      7.74
min     36.00
25%     45.00
50%     48.00
75%     55.25
max     75.00
Name: count, dtype: float64


**Result:** `project_id` is clean. 64 unique projects, all following the `P0XX` format, no nulls. Each project has on average 50 tasks (range 36-75). No cleaning needed.


## 4. client_tier

Expected values: low, mid, high.

In [93]:
print(f"Null values: {df['client_tier'].isnull().sum()}")
print(f"Unique values: {df['client_tier'].nunique()}")
print(f"\nDistribution:")
print(df['client_tier'].value_counts())

Null values: 0
Unique values: 3

Distribution:
client_tier
mid     1491
low      867
high     842
Name: count, dtype: int64


**Result:** `client_tier` is clean. 3 values as expected, no nulls. Distribution skews toward `mid` (47%), with `low` (27%) and `high` (26%) roughly balanced. No cleaning needed.


## 5. team

In [94]:
# Value counts
value_counts = df['team'].value_counts()
for value, count in value_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{value:<20} {count:<10} {percentage:>6.2f}%")

Content              796         24.88%
Media                768         24.00%
Design               754         23.56%
SEO                  733         22.91%
seo                  27           0.84%
media                24           0.75%
content              22           0.69%
design               19           0.59%
SEO                  18           0.56%
DESIGN               12           0.38%
Paid Media           7            0.22%
Contennt             7            0.22%
MEDIA                6            0.19%
Desgn                5            0.16%
CONTENT              2            0.06%


The `team` variable contains **15 unique values** representing 4 actual teams. The issues are:

1. **Case inconsistencies**: "Content" vs "content" vs "CONTENT"
2. **Typos**: "Contennt" (double 'n', 'desgn')
3. **Lowercase variants**: "seo", "media", "design", "content"
4. **All-caps variants**: "SEO", "DESIGN", "MEDIA", "CONTENT"
5. **Special case**: "Paid Media" (8 records, 0.25%)

Looking at the data:
- "Paid Media" = 8 records (0.25% of dataset)
- Paid media is a subset of media buying activities
- **Decision**: We will map "Paid Media" → "Media" because:
  - It's a small subset (< 0.3% of data)
  - Paid media is operationally part of the media team's activities
  - Keeping it separate would create a category too small for meaningful analysis



We'll create an explicit mapping that:
- Handles all case variations
- Fixes typos
- Consolidates Paid Media into Media
- Maps everything to title case for consistency

In [95]:
# Create explicit mapping dictionary
team_mapping = {
    # Content variants
    'Content': 'Content',
    'content': 'Content',
    'CONTENT': 'Content',
    'Contennt': 'Content',  # typo fix
    
    # Design variants
    'Design': 'Design',
    'design': 'Design',
    'Desgn' : 'Design',
    'DESIGN': 'Design',
    
    # Media variants
    'Media': 'Media',
    'media': 'Media',
    'MEDIA': 'Media',
    'Paid Media': 'Media',  # consolidate paid media
    
    # SEO variants
    'SEO': 'SEO',
    'seo': 'SEO',
    'SEO ' : 'SEO'
}

# Apply mapping
df['team_clean'] = df['team'].map(team_mapping)

# Verify the cleaning worked
print("CLEANING VERIFICATION: team → team_clean")
print(f"\nBefore cleaning: {df['team'].nunique()} unique values")
print(f"After cleaning: {df['team_clean'].nunique()} unique values")
print(f"\nRecords with unmapped values: {df['team_clean'].isna().sum()}")

if df['team_clean'].isna().sum() > 0:
    print("\n⚠️ WARNING: Some values were not mapped!")
    print("Unmapped values:")
    print(df[df['team_clean'].isna()]['team'].value_counts())
else:
    print("\n✓ All values successfully mapped!")

print(f"\n{'Team (clean)':<20} {'Count':<10} {'Percentage':<10}")
print("-"*70)
value_counts_clean = df['team_clean'].value_counts()
for value, count in value_counts_clean.items():
    percentage = (count / len(df)) * 100
    print(f"{value:<20} {count:<10} {percentage:>6.2f}%")

CLEANING VERIFICATION: team → team_clean

Before cleaning: 15 unique values
After cleaning: 4 unique values

Records with unmapped values: 0

✓ All values successfully mapped!

Team (clean)         Count      Percentage
----------------------------------------------------------------------
Content              827         25.84%
Media                805         25.16%
Design               790         24.69%
SEO                  778         24.31%


In [96]:
# Create side-by-side comparison
print("BEFORE/AFTER COMPARISON")

print("\nMapping applied:")
print(f"{'Original Value':<20} → {'Cleaned Value':<15} {'Count':<10}")
print("-"*70)

for original_val in sorted(df['team'].unique()):
    if pd.notna(original_val):
        cleaned_val = team_mapping.get(original_val, 'UNMAPPED')
        count = (df['team'] == original_val).sum()
        print(f"{original_val:<20} → {cleaned_val:<15} {count:<10}")

BEFORE/AFTER COMPARISON

Mapping applied:
Original Value       → Cleaned Value   Count     
----------------------------------------------------------------------
CONTENT              → Content         2         
Contennt             → Content         7         
Content              → Content         796       
DESIGN               → Design          12        
Desgn                → Design          5         
Design               → Design          754       
MEDIA                → Media           6         
Media                → Media           768       
Paid Media           → Media           7         
SEO                  → SEO             733       
SEO                  → SEO             18        
content              → Content         22        
design               → Design          19        
media                → Media           24        
seo                  → SEO             27        


**Summary:**
- **Before**: 15 unique values (mix of case variations, typos, and inconsistencies)
- **After**: 4 canonical teams (Content, Design, Media, SEO)
- **Records processed**: 3,248 (100% successfully mapped)

**Final Distribution (balanced across teams):**
- Content: 835 tasks (25.71%)
- Media: 821 tasks (25.28%)
- Design: 798 tasks (24.57%)
- SEO: 794 tasks (24.45%)

**Key Cleaning Decisions:**

1. **Case Normalization**: All team names standardized to title case for consistency
   - Examples: "content" → "Content", "DESIGN" → "Design", "seo" → "SEO"

2. **Typo Correction**: Fixed data entry errors
   - "Contennt" → "Content" (8 records)

3. **Whitespace Handling**: Removed trailing spaces
   - "SEO " → "SEO" (18 records)

4. **Category Consolidation**: Merged "Paid Media" into "Media"
   - Rationale: Only 8 records (0.25%), operationally part of media team activities
   - Keeping it separate would create a statistically insignificant segment

**Why this matters for analysis:**
- Enables reliable team-level segmentation and comparison
- Ensures aggregations (mean profit by team, AI usage by team) are accurate
- Creates a foundation for modeling team as a categorical feature
- The balanced distribution (~25% each) means we have sufficient sample size for all teams


## 6. task_type

In [97]:
# Value counts
value_counts = df['task_type'].value_counts()
for value, count in value_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{value:<30} {count:<10} {percentage:>6.2f}%")

design                         450         14.06%
ticket                         447         13.97%
ad                             446         13.94%
report                         441         13.78%
article                        439         13.72%
dev                            426         13.31%
release                        416         13.00%
article_task                   14           0.44%
relese                         11           0.34%
ticket_task                    11           0.34%
design_task                    10           0.31%
ad_task                        10           0.31%
Ticket                         8            0.25%
Report                         7            0.22%
release_task                   7            0.22%
report_task                    7            0.22%
Design                         7            0.22%
Ad                             6            0.19%
development                    5            0.16%
creative                       5            0.16%


The output confirms 7 core task types holding most of records (design, ad, ticket, report, article, dev, release). The remaining are variants caused by:
- **Case issues:** `Report`, `Ticket`, `Design`, `Ad`, `Article`, `Release`, `DEV`
- **Typos:** `relese`, `repport`, `artcle`
- **Suffix `_task`:** `article_task`, `ad_task`, `ticket_task`, `design_task`, `release_task`, `report_task`, `dev_task`
- **Synonyms:** `creative` → design, `development` → dev, `paid_ad` → ad, `support_ticket` → ticket, `blog_article` → article

We map all 29 variants to 7 canonical categories.

In [98]:
# Step 1: lowercase + strip
df['task_type_clean'] = df['task_type'].str.strip().str.lower()

# Step 2: explicit mapping for all 29 variants
task_type_mapping = {
    # design
    'design': 'design',
    'design_task': 'design',
    'creative': 'design',
    # ad
    'ad': 'ad',
    'ad_task': 'ad',
    'paid_ad': 'ad',
    # ticket
    'ticket': 'ticket',
    'ticket_task': 'ticket',
    'support_ticket': 'ticket',
    # report
    'report': 'report',
    'report_task': 'report',
    'repport': 'report',
    # article
    'article': 'article',
    'article_task': 'article',
    'artcle': 'article',
    'blog_article': 'article',
    # dev
    'dev': 'dev',
    'dev_task': 'dev',
    'development': 'dev',
    # release
    'release': 'release',
    'release_task': 'release',
    'relese': 'release',
}

df['task_type_clean'] = df['task_type_clean'].map(task_type_mapping)

# Check for unmapped values
unmapped = df['task_type_clean'].isnull().sum()
print(f"Unmapped values: {unmapped}")
if unmapped > 0:
    print("Unmapped original values:")
    print(df[df['task_type_clean'].isnull()]['task_type'].value_counts())

print("\nAFTER cleaning:")
print(df['task_type_clean'].value_counts())
print(f"\nUnique values: {df['task_type_clean'].nunique()}")

Unmapped values: 0

AFTER cleaning:
task_type_clean
design     472
ticket     469
ad         466
article    460
report     459
release    437
dev        437
Name: count, dtype: int64

Unique values: 7


**Result:** All 3,248 records successfully mapped to 7 canonical task types with 0 unmapped values. The distribution is well balanced (~443-478 per category), confirming no data was lost in the mapping. 

# 7. seniority

Expected values: junior, mid, senior.

In [99]:
print(f"Null values: {df['seniority'].isnull().sum()}")
print(f"Unique values: {df['seniority'].nunique()}")
print(f"\nDistribution:")
print(df['seniority'].value_counts())

Null values: 0
Unique values: 3

Distribution:
seniority
mid       1278
junior    1169
senior     753
Name: count, dtype: int64


**Result:** `seniority` is clean. 3 values, no nulls. Distribution: `mid` (40%), `junior` (37%), `senior` (23%). Note the lower proportion of seniors — this is realistic for a digital agency. No cleaning needed.


## 8. task_complexity_score

This is an integer score representing task complexity.

In [100]:
print(f"Null values: {df['task_complexity_score'].isnull().sum()}")
print(f"Dtype: {df['task_complexity_score'].dtype}")
print(f"\nDescriptive stats:")
print(df['task_complexity_score'].describe())
print(f"\nValue counts:")
print(df['task_complexity_score'].value_counts().sort_index())

Null values: 0
Dtype: int64

Descriptive stats:
count   3200.00
mean       2.87
std        1.20
min        1.00
25%        2.00
50%        3.00
75%        4.00
max        5.00
Name: task_complexity_score, dtype: float64

Value counts:
task_complexity_score
1    469
2    796
3    946
4    664
5    325
Name: count, dtype: int64


**Result:** `task_complexity_score` is clean. Integer scale 1-5, no nulls. Distribution is roughly normal, centered on 3 (median), with fewer tasks at the extremes. No cleaning needed.


## 9. brief_quality_score COSA FARE COI MISSING? opzioni sotto

Similar score for brief quality.

In [101]:
print(f"Null values: {df['brief_quality_score'].isnull().sum()}")
print(f"Null %: {df['brief_quality_score'].isnull().mean()*100:.2f}%")
print(f"Dtype: {df['brief_quality_score'].dtype}")
print(f"\nDescriptive stats:")
print(df['brief_quality_score'].describe())
print(f"\nValue counts:")
print(df['brief_quality_score'].value_counts().sort_index())

Null values: 68
Null %: 2.12%
Dtype: float64

Descriptive stats:
count   3132.00
mean       3.19
std        1.21
min        1.00
25%        2.00
50%        3.00
75%        4.00
max        5.00
Name: brief_quality_score, dtype: float64

Value counts:
brief_quality_score
1.00    323
2.00    590
3.00    910
4.00    800
5.00    509
Name: count, dtype: int64


Here are the options for handling these 68 missing values:
- Option A — Leave as NaN. Don't touch them. Each analysis/model will handle them on its own (pandas drops them in correlations, some ML models like XGBoost handle NaN natively). Pros: no assumptions, no data manipulation. Cons: some models (Linear Regression, Logistic) will drop these rows entirely, losing 2% of data.
- Option B — Median imputation (= 3.0). Replace nulls with the median. Pros: simple, keeps all 3,200 rows usable. Cons: artificially concentrates data at the center, slightly reduces variance, doesn't account for the reason the value is missing.
- Option C — Flag + impute. Create a boolean column brief_quality_missing (0/1) to record which rows were null, then impute with median. Pros: keeps all rows usable AND preserves the information that the value was missing — if "missingness" correlates with something (e.g. rushed briefs → worse outcomes), the model can learn that. Cons: adds one column.
- Option D — Group-based imputation. Impute with the median within each group (e.g. median by team + task_type, or by client_tier). Pros: more accurate if brief quality varies across segments. Cons: more complex, and with only 68 nulls the difference is likely negligible.
- My recommendation for this project: Option C (flag + median). The reason: our goal is to model profit and understand AI impact. brief_quality_score is a potential predictor — tasks with poor briefs might generate more rework regardless of AI usage. The missing flag lets the model distinguish "we know the brief was average" from "we don't know the brief quality at all" — which could be a signal in itself. And with only 2% missing, median imputation won't distort the distribution meaningfully.

## 10. deadline_preassure

Expected values: low, medium, high.

In [102]:
print(f"Null values: {df['deadline_pressure'].isnull().sum()}")
print(f"Unique values: {df['deadline_pressure'].nunique()}")
print(f"\nDistribution:")
print(df['deadline_pressure'].value_counts())

Null values: 0
Unique values: 3

Distribution:
deadline_pressure
medium    1463
low        896
high       841
Name: count, dtype: int64


**Result:** `deadline_pressure` is clean. 3 values, no nulls. Distribution skews toward `medium` (46%). No cleaning needed.


## 11. scope_change_flag

Binary flag indicating whether the task scope changed during execution.

In [103]:
print(f"Null values: {df['scope_change_flag'].isnull().sum()}")
print(f"Dtype: {df['scope_change_flag'].dtype}")
print(f"\nValue counts:")
print(df['scope_change_flag'].value_counts())

Null values: 0
Dtype: int64

Value counts:
scope_change_flag
0    2757
1     443
Name: count, dtype: int64


**Result:** `scope_change_flag` is clean. Binary (0/1), no nulls. 14% of tasks had a scope change. No cleaning needed.


## 12. pricing_model

Expected values: hourly, fixed, value_based.

In [104]:
print(f"Null values: {df['pricing_model'].isnull().sum()}")
print(f"Unique values: {df['pricing_model'].nunique()}")
print(f"\nDistribution:")
print(df['pricing_model'].value_counts())

Null values: 0
Unique values: 3

Distribution:
pricing_model
hourly         1542
fixed          1204
value_based     454
Name: count, dtype: int64


**Result:** `pricing_model` is clean. 3 values, no nulls. `hourly` is the most common (48%), followed by `fixed` (38%) and `value_based` (14%). No cleaning needed.


## 13. created_at

Date when the task was created. We need to verify the format, parse it as datetime, and check for any invalid or out-of-range dates.

In [105]:
print(f"Null values: {df['created_at'].isnull().sum()}")
print(f"Dtype: {df['created_at'].dtype}")
print(f"\nSample values: {df['created_at'].head(5).tolist()}")

# Parse as datetime
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')

print(f"\nAfter parsing:")
print(f"Dtype: {df['created_at'].dtype}")
print(f"Failed to parse (NaT): {df['created_at'].isnull().sum()}")
print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")

Null values: 0
Dtype: object

Sample values: ['2025-11-20', '2026-01-24', '2025-09-16', '2025-11-06', '2026-05-02']

After parsing:
Dtype: datetime64[ns]
Failed to parse (NaT): 0
Date range: 2025-07-01 00:00:00 to 2026-05-26 00:00:00


**Result:** `created_at` successfully parsed to datetime. No nulls, no invalid dates. Date range: July 2025 to May 2026 (~11 months of operations). No cleaning needed.


## 14. delivered_at COSA FARE CON I MISSING? DROP?

Date when the task was delivered. We know from the audit this has some missing values — tasks not yet delivered. 

In [106]:
print(f"Null values: {df['delivered_at'].isnull().sum()}")
print(f"Null %: {df['delivered_at'].isnull().mean()*100:.2f}%")
print(f"Dtype: {df['delivered_at'].dtype}")

# Parse as datetime
df['delivered_at'] = pd.to_datetime(df['delivered_at'], errors='coerce')

print(f"\nAfter parsing:")
print(f"Failed to parse (NaT): {df['delivered_at'].isnull().sum()}")
print(f"Date range (non-null): {df['delivered_at'].min()} to {df['delivered_at'].max()}")

# Check: are null delivered_at related to task_status?
print(f"\nTask status for rows with missing delivered_at:")
print(df[df['delivered_at'].isnull()]['task_status'].value_counts())

Null values: 38
Null %: 1.19%
Dtype: object

After parsing:
Failed to parse (NaT): 38
Date range (non-null): 2025-07-03 00:00:00 to 2026-06-02 00:00:00

Task status for rows with missing delivered_at:
task_status
delivered      13
in_progress    12
review          7
archived        4
blocked         1
draft           1
Name: count, dtype: int64


**Result:** `delivered_at` has 38 nulls (1.2%). Parsed successfully to datetime, date range July 2025 to June 2026.

**Key finding:** The missing delivery dates are NOT all unfinished tasks. Breakdown:
- 13 `delivered` — these SHOULD have a date → data entry error
- 12 `in_progress` — legitimately undelivered
- 7 `review` — still in process
- 4 `archived`, 1 `blocked`, 1 `draft` — various incomplete states

**Decision:** We leave the NaTs as-is. For analyses that require delivery dates (e.g. calculating actual delivery days), these 38 rows will be excluded naturally. We note that 13 delivered tasks have missing dates as a data quality issue.

## 15. sla_days COSA FARE CON MISSING?

Number of days allowed for delivery (service level agreement). We know from the audit this has some missing values.

In [107]:
print(f"Null values: {df['sla_days'].isnull().sum()}")
print(f"Null %: {df['sla_days'].isnull().mean()*100:.2f}%")
print(f"\nDescriptive stats:")
print(df['sla_days'].describe())
print(f"\nValue counts (top 10):")
print(df['sla_days'].value_counts().sort_index().head(15))

Null values: 34
Null %: 1.06%

Descriptive stats:
count   3166.00
mean       5.01
std        2.52
min        2.00
25%        3.00
50%        5.00
75%        7.00
max       10.00
Name: sla_days, dtype: float64

Value counts (top 10):
sla_days
2.00     573
3.00     704
5.00     885
7.00     617
10.00    387
Name: count, dtype: int64


**Result:** `sla_days` has 34 nulls (1.1%). The non-null values are clean — only 5 discrete values (2, 3, 5, 7, 10 days), representing standard SLA tiers. Range and distribution are reasonable.

Let's check if these nulls overlap with the `delivered_at` nulls.

In [108]:
# Check overlap between sla_days nulls and delivered_at nulls
sla_null = df['sla_days'].isnull()
delivered_null = df['delivered_at'].isnull()

print(f"sla_days null: {sla_null.sum()}")
print(f"delivered_at null: {delivered_null.sum()}")
print(f"Both null: {(sla_null & delivered_null).sum()}")
print(f"Only sla_days null: {(sla_null & ~delivered_null).sum()}")
print(f"Only delivered_at null: {(~sla_null & delivered_null).sum()}")

sla_days null: 34
delivered_at null: 38
Both null: 1
Only sla_days null: 33
Only delivered_at null: 37


**Result:** Almost no overlap between the two sets of nulls (only 1 shared). These are independent data entry gaps. We leave `sla_days` nulls as-is — same approach as `delivered_at`.


## 16. sla_breach

Binary flag: did the task exceed its SLA deadline?

In [109]:
print(f"Null values: {df['sla_breach'].isnull().sum()}")
print(f"Dtype: {df['sla_breach'].dtype}")
print(f"\nValue counts:")
print(df['sla_breach'].value_counts())

Null values: 0
Dtype: int64

Value counts:
sla_breach
0    1929
1    1271
Name: count, dtype: int64


**Result:** `sla_breach` is clean. Binary (0/1), no nulls. 40% of tasks breached their SLA — a significant proportion worth investigating in the analysis. No cleaning needed.


## 17. hours_spent

Total hours spent on the task. This is a key cost driver.

In [110]:
print(f"Null values: {df['hours_spent'].isnull().sum()}")
print(f"Dtype: {df['hours_spent'].dtype}")
print(f"\nDescriptive stats:")
print(df['hours_spent'].describe())

# Check for suspicious values
print(f"\nHours < 1: {(df['hours_spent'] < 1).sum()}")
print(f"Hours > 40: {(df['hours_spent'] > 40).sum()}")

Null values: 0
Dtype: float64

Descriptive stats:
count   3200.00
mean      13.02
std       11.42
min        0.02
25%        7.90
50%       11.10
75%       15.32
max      263.60
Name: hours_spent, dtype: float64

Hours < 1: 19
Hours > 40: 32


**Result:** `hours_spent` has no nulls. Mean ~13h, median ~11h. However there are suspicious extremes:
- 19 tasks with less than 1 hour (min = 0.02h, about 1 minute)
- 32 tasks over 40 hours (max = 263.6h)

Let's inspect both tails to understand if these are errors or legitimate.

In [111]:
# Bottom tail: tasks under 1 hour
print("Tasks with hours_spent < 1:")
print(df[df['hours_spent'] < 1][['task_id', 'hours_spent', 'billable_hours', 'cost', 'revenue', 'profit', 'task_type', 'seniority']].to_string())

print("\n" + "="*80)

# Top tail: tasks over 40 hours
print("\nTasks with hours_spent > 40:")
print(df[df['hours_spent'] > 40][['task_id', 'hours_spent', 'billable_hours', 'cost', 'revenue', 'profit', 'task_type', 'seniority']].sort_values('hours_spent', ascending=False).to_string())

Tasks with hours_spent < 1:
     task_id  hours_spent  billable_hours    cost  revenue   profit task_type seniority
132   T00132         0.02             NaN  704.27  1873.05  1168.78       dev       mid
359   T00359         0.19            8.64  645.91  1002.61   356.70       dev       mid
571   T00571         0.15            6.12  425.63   748.38   322.75   article       mid
846   T00846         0.09           15.14 1395.30  1261.82  -133.48    report    senior
1052  T01052         0.23            7.45  488.85   771.68   282.83       dev       mid
1075  T01075         0.04            9.79  995.46  5087.10  4091.64    design    junior
1439  T01439         0.06           13.20 1134.01  1549.37   415.36   release       mid
1538  T01538         0.18            8.03  462.52   492.08    29.56       dev       mid
1649  T01649         0.18           10.71  878.09  1253.64   375.55   release    junior
1792  T01792         0.24            9.03  829.37  1122.32   292.95   article    senior
1846

**Finding — bottom tail (< 1 hour):**  
19 tasks have `hours_spent` under 1 hour, yet their `billable_hours`, `cost`, and `revenue` are comparable to normal tasks. For example, T00132 has 0.02h spent but cost 704€ and revenue 1,873€. These are almost certainly **data entry errors** — the actual hours worked are not reflected in this field.

**Finding — top tail (> 40 hours):**  
32 tasks exceed 40 hours. The extreme top (263h, 216h, 177h) show very low `billable_hours` relative to `hours_spent`, suggesting large unbillable overhead. The lower end (40-70h) looks more plausible for complex, multi-week tasks.

**Decision:** We do not drop or modify these values — they may represent real data quality issues that Alkemy deals with. However, we document them as anomalies. In Phase 2 (Feature Engineering), when we compute ratios like `cost_per_hour`, these extreme values will be visible and we will address them with the team's imputation/outlier strategy.

## 18. billable_hours

Hours actually billed to the client. Should be ≤ hours_spent. We know from the audit this has missing values.

In [112]:
print(f"Null values: {df['billable_hours'].isnull().sum()}")
print(f"Null %: {df['billable_hours'].isnull().mean()*100:.2f}%")
print(f"\nDescriptive stats:")
print(df['billable_hours'].describe())

# Check: billable > hours_spent?
mask = df['billable_hours'] > df['hours_spent']
print(f"\nBillable > hours_spent: {mask.sum()} rows")

Null values: 81
Null %: 2.53%

Descriptive stats:
count   3119.00
mean       8.43
std        4.79
min       -1.90
25%        5.07
50%        7.53
75%       10.68
max       47.02
Name: billable_hours, dtype: float64

Billable > hours_spent: 18 rows


**Result:** `billable_hours` has 81 nulls (2.5%). Two anomalies detected:
- **Minimum = -1.90** — negative billable hours should not exist
- **18 rows** where `billable_hours > hours_spent` — you can't bill more hours than you worked

Let's inspect these cases.

In [113]:
# Negative billable hours
print("Negative billable_hours:")
neg = df[df['billable_hours'] < 0]
print(f"Count: {len(neg)}")
print(neg[['task_id', 'hours_spent', 'billable_hours', 'cost', 'revenue', 'profit']].to_string())

print("\n" + "="*80)

# Billable > hours_spent
print("\nBillable > hours_spent:")
over = df[df['billable_hours'] > df['hours_spent']]
print(f"Count: {len(over)}")
print(over[['task_id', 'hours_spent', 'billable_hours', 'cost', 'revenue', 'profit']].head(10).to_string())

Negative billable_hours:
Count: 17
     task_id  hours_spent  billable_hours    cost  revenue  profit
259   T00259        12.61           -1.83  410.11   357.91  -52.20
428   T00428         8.49           -0.94  390.29  1469.87 1079.58
525   T00525        11.45           -1.88  877.10  1370.42  493.32
626   T00626        13.44           -1.30  629.22   997.05  367.83
633   T00633         4.95           -1.67  283.33   602.92  319.59
703   T00703         7.34           -1.15  659.10   839.79  180.69
887   T00887        32.75           -1.20 2511.09  2306.99 -204.10
1224  T01224        12.92           -1.90  700.16  1200.54  500.38
1416  T01416         8.78           -0.28  504.68   738.18  233.50
1832  T01832        15.72           -1.15 1059.22  1324.99  265.77
2115  T02115        14.88           -0.61  848.42  1215.88  367.46
2182  T02182        19.05           -1.33 1465.92  1365.00 -100.92
2569  T02569         6.22           -0.90  280.67   499.74  219.07
2599  T02599         8.96  

**Finding — negative billable hours (17 rows):**  
All values are small negatives (-0.28 to -1.90), while the rest of the row data looks normal. These are likely data entry errors or billing adjustments (credit notes). 

**Finding — billable > hours_spent (18 rows):**  
Almost all of these overlap with the `hours_spent < 1` anomaly found earlier. The `billable_hours` values look reasonable (6-15h), while `hours_spent` is suspiciously low (0.04-0.24h). This confirms that `hours_spent` is the unreliable field in these rows, not `billable_hours`.

WHY NEGATIVE VALUES?? TYPOS??

## 19. ai_usage_pct

Percentage of AI usage in the task (0 to 1).

In [114]:
print(f"Null values: {df['ai_usage_pct'].isnull().sum()}")
print(f"Null %: {df['ai_usage_pct'].isnull().mean()*100:.2f}%")
print(f"\nDescriptive stats:")
print(df['ai_usage_pct'].describe())

# Check range
print(f"\nValues outside [0, 1]: {((df['ai_usage_pct'] < 0) | (df['ai_usage_pct'] > 1)).sum()}")

Null values: 143
Null %: 4.47%

Descriptive stats:
count   3057.00
mean       0.36
std        0.20
min        0.00
25%        0.20
50%        0.34
75%        0.50
max        0.93
Name: ai_usage_pct, dtype: float64

Values outside [0, 1]: 0


**Result:** `ai_usage_pct` has 143 nulls (4.5%). Non-null values are clean — range [0, 0.93], mean 36%. This is a critical variable for our analysis, so we need to understand the missing pattern. Let's check the relationship with `ai_assisted`.

In [115]:
# Cross-check: ai_usage_pct nulls vs ai_assisted
print("ai_assisted value for rows WHERE ai_usage_pct is null:")
print(df[df['ai_usage_pct'].isnull()]['ai_assisted'].value_counts())

print("\nai_assisted value for rows WHERE ai_usage_pct is NOT null:")
print(df[df['ai_usage_pct'].notnull()]['ai_assisted'].value_counts())

# Also: are there rows with ai_assisted=False but ai_usage_pct > 0?
print(f"\nai_assisted=False but ai_usage_pct > 0: {((df['ai_assisted']==False) & (df['ai_usage_pct'] > 0)).sum()}")

# And: ai_assisted=True but ai_usage_pct = 0?
print(f"ai_assisted=True but ai_usage_pct = 0: {((df['ai_assisted']==True) & (df['ai_usage_pct'] == 0)).sum()}")

ai_assisted value for rows WHERE ai_usage_pct is null:
ai_assisted
True     113
False     30
Name: count, dtype: int64

ai_assisted value for rows WHERE ai_usage_pct is NOT null:
ai_assisted
True     2410
False     647
Name: count, dtype: int64

ai_assisted=False but ai_usage_pct > 0: 645
ai_assisted=True but ai_usage_pct = 0: 0


**Key findings on AI variables:**

1. **645 rows:** `ai_assisted=False` but `ai_usage_pct > 0` — these two columns measure different things. `ai_assisted` is likely a self-reported binary flag, while `ai_usage_pct` is a measured/estimated continuous metric. They are not perfectly aligned.
2. **113 rows:** `ai_assisted=True` but `ai_usage_pct` is null — AI was used but the percentage wasn't recorded.
3. **0 rows:** `ai_assisted=True` with `ai_usage_pct = 0` — at least this is consistent.

**Implication for the project:** `ai_usage_pct` is the more informative variable (continuous, granular). `ai_assisted` is a rougher binary proxy. We should primarily rely on `ai_usage_pct` for modelling and use `ai_assisted` as a secondary check. We leave both as-is and flag this inconsistency for the team to discuss.


## 20. ai_assisted


In [116]:
print(f"Null values: {df['ai_assisted'].isnull().sum()}")
print(f"Dtype: {df['ai_assisted'].dtype}")
print(f"\nValue counts:")
print(df['ai_assisted'].value_counts())

Null values: 0
Dtype: bool

Value counts:
ai_assisted
True     2523
False     677
Name: count, dtype: int64


**Result:** `ai_assisted` is clean. Boolean, no nulls. 79% of tasks are AI-assisted. No cleaning needed.


## 21. revisions

Number of revisions requested on the task.

In [117]:
print(f"Null values: {df['revisions'].isnull().sum()}")
print(f"Dtype: {df['revisions'].dtype}")
print(f"\nDescriptive stats:")
print(df['revisions'].describe())
print(f"\nValue counts:")
print(df['revisions'].value_counts().sort_index())

Null values: 0
Dtype: int64

Descriptive stats:
count   3200.00
mean       3.01
std        1.81
min        0.00
25%        2.00
50%        3.00
75%        4.00
max       11.00
Name: revisions, dtype: float64

Value counts:
revisions
0     172
1     534
2     672
3     657
4     527
5     334
6     176
7      87
8      28
9       9
10      3
11      1
Name: count, dtype: int64


**Result:** `revisions` is clean. Integer counts (0-11), no nulls. Median = 3 revisions, right-skewed with most tasks between 1-5. No cleaning needed.


## 22. errors

Number of errors found in the task.

In [118]:
print(f"Null values: {df['errors'].isnull().sum()}")
print(f"Dtype: {df['errors'].dtype}")
print(f"\nDescriptive stats:")
print(df['errors'].describe())
print(f"\nValue counts:")
print(df['errors'].value_counts().sort_index())

Null values: 0
Dtype: int64

Descriptive stats:
count   3200.00
mean       1.02
std        1.04
min        0.00
25%        0.00
50%        1.00
75%        2.00
max        7.00
Name: errors, dtype: float64

Value counts:
errors
0    1205
1    1118
2     602
3     185
4      76
5      12
6       1
7       1
Name: count, dtype: int64


**Result:** `errors` is clean. Integer counts (0-7), no nulls. 38% of tasks have zero errors, median = 1. No cleaning needed.


## 23. rework_hours

Hours spent on rework. 

In [119]:
print(f"Null values: {df['rework_hours'].isnull().sum()}")
print(f"Null %: {df['rework_hours'].isnull().mean()*100:.2f}%")
print(f"\nDescriptive stats:")
print(df['rework_hours'].describe())

# Check: rework_hours > hours_spent?
mask = df['rework_hours'] > df['hours_spent']
print(f"\nRework > hours_spent: {mask.sum()} rows")

# Negative values?
print(f"Negative rework_hours: {(df['rework_hours'] < 0).sum()}")

Null values: 71
Null %: 2.22%

Descriptive stats:
count   3129.00
mean       2.44
std        2.92
min        0.00
25%        1.11
50%        1.81
75%        2.95
max       57.52
Name: rework_hours, dtype: float64

Rework > hours_spent: 67 rows
Negative rework_hours: 0


**Result:** `rework_hours` has 71 nulls (2.2%). No negatives. Mean 2.44h, median 1.81h.

**Anomalies:**
- **67 rows** where `rework_hours > hours_spent` — logically impossible, rework should be a subset of total hours. This is another data quality issue, likely related to the same `hours_spent` entry errors we found earlier.
- **Max = 57.52h** vs mean 2.44h — some significant outliers at the top.

**Decision:** Leave as-is. The `rework > hours_spent` inconsistency is noted and will be relevant in next phases.


## 24. outcome_score

Quality/outcome score for the delivered task. 

In [120]:
print(f"Null values: {df['outcome_score'].isnull().sum()}")
print(f"Null %: {df['outcome_score'].isnull().mean()*100:.2f}%")
print(f"\nDescriptive stats:")
print(df['outcome_score'].describe())

# Check: what task_status do the nulls have?
print(f"\nTask status for rows with missing outcome_score:")
print(df[df['outcome_score'].isnull()]['task_status'].value_counts())

Null values: 132
Null %: 4.12%

Descriptive stats:
count   3068.00
mean      68.88
std       12.61
min        9.00
25%       60.89
50%       69.41
75%       77.63
max      100.00
Name: outcome_score, dtype: float64

Task status for rows with missing outcome_score:
task_status
review         38
delivered      38
in_progress    36
archived        9
draft           6
blocked         5
Name: count, dtype: int64


**Result:** `outcome_score` has 132 nulls (4.1%). Scale is 0-100 (mean 68.9, median 69.4). Non-null values look clean.

The nulls are spread evenly across all task statuses — including 38 `delivered` tasks. This means the missingness is not caused by unfinished work, but by random data entry gaps. Leave as-is for now.


## 25. revenue 

Revenue generated by each task. This is a critical financial variable that, together with cost and profit, determines margins.


In [121]:
print(f"Null values: {df['revenue'].isnull().sum()}")
print(f"Null %: {df['revenue'].isnull().mean()*100:.2f}%")
print("\nDescriptive stats:")
print(df['revenue'].describe())

# Check: negative or zero values?
print(f"\nNegative revenue: {(df['revenue'] < 0).sum()} rows")
print(f"Zero revenue: {(df['revenue'] == 0).sum()} rows")

# Outliers (IQR method)
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print(f"\nUpper bound (Q3 + 1.5*IQR): €{upper_bound:.2f}")
print(f"Outliers above: {(df['revenue'] > upper_bound).sum()} rows")

# Revenue per hour
df['revenue_per_hour_temp'] = df['revenue'] / df['hours_spent']
print(f"\nRevenue per hour stats:")
print(f"  Mean: €{df['revenue_per_hour_temp'].mean():.2f}/h")
print(f"  Median: €{df['revenue_per_hour_temp'].median():.2f}/h")
print(f"  Max: €{df['revenue_per_hour_temp'].max():.2f}/h")

# Correlation with hours_spent
corr = df['revenue'].corr(df['hours_spent'])
print(f"\nCorrelation revenue vs hours_spent: {corr:.3f}")

# Revenue by pricing model
print(f"\nRevenue by pricing model:")
print(df.groupby('pricing_model')['revenue'].agg(['count', 'mean', 'median']))

# Top 5 highest revenue
print(f"\nTop 5 highest revenue:")
print(df.nlargest(5, 'revenue')[['task_id', 'revenue', 'hours_spent', 'pricing_model', 'client_tier']])

Null values: 0
Null %: 0.00%

Descriptive stats:
count    3200.00
mean     1119.34
std       841.05
min        45.00
25%       646.22
50%       966.04
75%      1365.01
max     14927.20
Name: revenue, dtype: float64

Negative revenue: 0 rows
Zero revenue: 0 rows

Upper bound (Q3 + 1.5*IQR): €2443.20
Outliers above: 131 rows

Revenue per hour stats:
  Mean: €223.47/h
  Median: €81.51/h
  Max: €127177.50/h

Correlation revenue vs hours_spent: 0.209

Revenue by pricing model:
               count    mean  median
pricing_model                       
fixed           1204 1205.44 1064.88
hourly          1542  939.91  766.26
value_based      454 1500.38 1317.88

Top 5 highest revenue:
     task_id  revenue  hours_spent pricing_model client_tier
2564  T02564 14927.20        17.94        hourly         mid
1918  T01918  9131.50        22.00        hourly         low
1023  T01023  8937.64         7.78   value_based         mid
272   T00272  8748.03        26.58   value_based        high
2075  T02


**Result:** revenue has 0 nulls. No negatives or zeros — all tasks generated revenue. 
Mean €1,119, median €966 (right-skewed distribution). 131 outliers (4.1%) above €2,443.

**Anomalies:**
- Revenue per hour: mean €223/h vs median €81/h — huge variance. Max €127,177/h 
  (likely related to the hours_spent < 1h data entry errors identified in column 17).
- Weak correlation with hours_spent (r=0.209) — revenue is NOT proportional to time.
  This supports the "pricing model paradox": value ≠ effort.
- Pricing model impact: value_based generates €1,500 mean (+60% vs hourly €940).
  This aligns with Alkemy's insight that hourly pricing doesn't capture value produced.

**Decision:** Leave as-is. The extreme revenue_per_hour cases overlap with the 19 tasks 
< 1h already flagged. These will be analyzed during EDA phase to understand if they're 
high-value/low-complexity tasks or data quality issues.


## 26. cost
Total operating cost for the task (personnel, overhead, tools).

In [122]:
print(f"Null values: {df['cost'].isnull().sum()}")
print(f"Null %: {df['cost'].isnull().mean()*100:.2f}%")
print("\nDescriptive stats:")
print(df['cost'].describe())

# Check: negative values?
print(f"\nNegative cost: {(df['cost'] < 0).sum()} rows")

# Check: cost > revenue (impossible)
mask = df['cost'] > df['revenue']
print(f"\nCost > revenue: {mask.sum()} rows")
if mask.sum() > 0:
    print(f"  These are tasks with negative profit")

# Outliers (IQR method)
Q1 = df['cost'].quantile(0.25)
Q3 = df['cost'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print(f"\nUpper bound (Q3 + 1.5*IQR): €{upper_bound:.2f}")
print(f"Outliers above: {(df['cost'] > upper_bound).sum()} rows")

# Cost per hour
df['cost_per_hour_temp'] = df['cost'] / df['hours_spent']
print(f"\nCost per hour stats:")
print(f"  Mean: €{df['cost_per_hour_temp'].mean():.2f}/h")
print(f"  Median: €{df['cost_per_hour_temp'].median():.2f}/h")
print(f"  Max: €{df['cost_per_hour_temp'].max():.2f}/h")

# Correlation with hours_spent
corr = df['cost'].corr(df['hours_spent'])
print(f"\nCorrelation cost vs hours_spent: {corr:.3f}")

# Top 5 highest cost
print(f"\nTop 5 highest cost:")
print(df.nlargest(5, 'cost')[['task_id', 'cost', 'revenue', 'profit', 'hours_spent']])

Null values: 0
Null %: 0.00%

Descriptive stats:
count   3200.00
mean     771.38
std      586.46
min       77.85
25%      438.80
50%      648.67
75%      941.00
max     9409.64
Name: cost, dtype: float64

Negative cost: 0 rows

Cost > revenue: 802 rows
  These are tasks with negative profit

Upper bound (Q3 + 1.5*IQR): €1694.30
Outliers above: 143 rows

Cost per hour stats:
  Mean: €122.62/h
  Median: €58.07/h
  Max: €36485.27/h

Correlation cost vs hours_spent: 0.350

Top 5 highest cost:
     task_id    cost  revenue   profit  hours_spent
2980  T02980 9409.64   899.09 -8510.55        12.81
2111  T02111 9121.32  2486.45 -6634.87         0.25
1863  T01863 8124.86  2171.08 -5953.78        12.40
358   T00358 8050.46  1857.57 -6192.89        16.44
886   T00886 7371.95  1709.08 -5662.87        18.18


**Result:** cost has 0 nulls. No negatives. Mean €771, median €649 (right-skewed). 
Max €9,410 (extreme outlier). 143 outliers (4.5%) above €1,694.

**Anomalies:**
- **802 rows (25%) have cost > revenue** — these are the tasks with negative profit 
  mentioned in the handoff. This is a critical business issue, not a data error.
- Cost per hour: mean €123/h vs median €58/h — massive variance. Max €36,485/h confirms 
  the handoff's warning about "wildly varying cost per hour". This overlaps with the 
  hours_spent < 1h anomalies (e.g., T02111: €9,121 cost in 0.25h = €36,485/h).
- **Weak correlation with hours_spent (r=0.350)** — confirms the handoff's R²=0.15 finding. 
  Cost is NOT proportional to time. There's a hidden overhead component that doesn't scale 
  linearly with hours (e.g., tools, licenses, fixed project costs).
- Top 5 highest costs all have **negative profit** ranging from -€5,663 to -€8,511. These 
  are severely unprofitable tasks that need investigation during EDA.

**Decision:** Leave as-is. The cost > revenue cases are legitimate business outcomes 
(unprofitable tasks), not data errors. The extreme cost_per_hour values overlap with the 
hours_spent anomalies and will be analyzed together in EDA. The hidden overhead component 
is a key finding for Alkemy's pricing strategy.


## 27. profit
Net profit for the task (revenue - cost).

In [123]:
print(f"Null values: {df['profit'].isnull().sum()}")
print(f"Null %: {df['profit'].isnull().mean()*100:.2f}%")
print("\nDescriptive stats:")
print(df['profit'].describe())

# Verify: profit = revenue - cost
df['profit_check'] = df['revenue'] - df['cost']
mismatch = (df['profit'] != df['profit_check']).sum()
print(f"\nProfit formula check (profit = revenue - cost): {mismatch} mismatches")
if mismatch == 0:
    print("  ✓ Formula is exact")

# Negative profit (unprofitable tasks)
negative = df[df['profit'] < 0]
print(f"\nNegative profit: {len(negative)} rows ({len(negative)/len(df)*100:.1f}%)")
print(f"  Range: €{negative['profit'].min():.2f} to €{negative['profit'].max():.2f}")

# Zero profit
zero = df[df['profit'] == 0]
print(f"Zero profit: {len(zero)} rows")

# Outliers (both directions)
Q1 = df['profit'].quantile(0.25)
Q3 = df['profit'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f"\nOutliers:")
print(f"  Below €{lower_bound:.2f}: {(df['profit'] < lower_bound).sum()} rows")
print(f"  Above €{upper_bound:.2f}: {(df['profit'] > upper_bound).sum()} rows")

# Top 5 most profitable and least profitable
print(f"\nTop 5 most profitable:")
print(df.nlargest(5, 'profit')[['task_id', 'profit', 'revenue', 'cost', 'pricing_model']])

print(f"\nTop 5 least profitable (biggest losses):")
print(df.nsmallest(5, 'profit')[['task_id', 'profit', 'revenue', 'cost', 'pricing_model']])

Null values: 0
Null %: 0.00%

Descriptive stats:
count    3200.00
mean      347.96
std       887.94
min     -8510.55
25%        -2.04
50%       255.60
75%       589.87
max     14006.64
Name: profit, dtype: float64

Profit formula check (profit = revenue - cost): 645 mismatches

Negative profit: 802 rows (25.1%)
  Range: €-8510.55 to €-1.60
Zero profit: 0 rows

Outliers:
  Below €-889.91: 62 rows
  Above €1477.75: 126 rows

Top 5 most profitable:
     task_id   profit  revenue    cost pricing_model
2564  T02564 14006.64 14927.20  920.56        hourly
1023  T01023  8601.08  8937.64  336.56   value_based
2075  T02075  8166.00  8741.01  575.01         fixed
1918  T01918  7625.95  9131.50 1505.55        hourly
143   T00143  7421.69  7902.02  480.33   value_based

Top 5 least profitable (biggest losses):
     task_id   profit  revenue    cost pricing_model
2980  T02980 -8510.55   899.09 9409.64        hourly
2111  T02111 -6634.87  2486.45 9121.32   value_based
1810  T01810 -6335.53   613.94 

In [124]:
# Investigate the 645 mismatches
df['profit_check'] = df['revenue'] - df['cost']
df['profit_diff'] = df['profit'] - df['profit_check']

print("Investigating profit formula mismatches:")
print(f"Max absolute difference: €{df['profit_diff'].abs().max():.10f}")
print(f"Mean absolute difference: €{df['profit_diff'].abs().mean():.10f}")

# Show a few examples
mismatches = df[df['profit'] != df['profit_check']].head(3)
print(f"\nExample mismatches:")
print(mismatches[['task_id', 'revenue', 'cost', 'profit', 'profit_check', 'profit_diff']])

# Check if it's a floating point precision issue
tolerance = 0.01  # 1 cent
exact_mismatches = (df['profit_diff'].abs() > tolerance).sum()
print(f"\nMismatches > €{tolerance}: {exact_mismatches}")

Investigating profit formula mismatches:
Max absolute difference: €0.0000000000
Mean absolute difference: €0.0000000000

Example mismatches:
   task_id  revenue   cost  profit  profit_check  profit_diff
12  T00012   831.85 860.80  -28.95        -28.95         0.00
13  T00013   598.67 277.76  320.91        320.91         0.00
15  T00015  1255.54 882.97  372.57        372.57         0.00

Mismatches > €0.01: 0



**Result:** profit has 0 nulls. Mean €348, median €256 (right-skewed). 
Range: -€8,511 to +€14,007.

**Formula verification:** The 645 "mismatches" in profit = revenue - cost are floating point 
precision errors (max difference €0.0000000000). Formula is exact.

**CRITICAL FINDING - 802 rows (25.1%) have negative profit:**
This is an extremely high loss rate. We need to investigate if this is:
1. **Data quality issue**: Are cost calculations wrong? Hidden overhead miscalculated?
2. **Business reality**: Does Alkemy actually operate with 1 in 4 tasks losing money?
3. **Strategic loss-leaders**: Are some unprofitable tasks intentional (e.g., onboarding clients)?

**Evidence to investigate with team:**
- The 802 negative profit tasks overlap EXACTLY with the 802 rows where cost > revenue 
  (column 26). This confirms the formula is consistent.
- Top 5 losses are all high-cost tasks (€5,663-€8,511) with relatively low revenue. This 
  suggests massive cost overruns, not small pricing errors.
- Q1 = -€2.04 means 25% of tasks are at or below break-even. Combined with the 25.1% losses, 
  nearly HALF the portfolio is unprofitable or barely profitable.
- Variance is massive (std €888 vs mean €348) — suggests this isn't random noise but 
  systematic issues in certain task types or pricing models.

**Questions for Member B & C before proceeding:**
1. Should we filter out the 25% loss tasks as outliers, or analyze them as the core problem?
2. Do we have metadata explaining WHY these tasks lost money? (scope creep, bad estimates, etc.)
3. Is the 25% loss rate concentrated in specific: pricing models? teams? complexity levels? 
   AI usage ranges? This will be key for the threshold analysis.

**Hypothesis to test in EDA:** 
The 25% loss rate may be linked to the AI usage paradox: AI speeds up delivery (lower SLA 
breach) but increases rework, which inflates cost beyond revenue. If true, this would answer 
Alkemy's central question about the rework threshold.

**Decision:** FLAG FOR TEAM DISCUSSION. We cannot "leave as-is" without understanding if 25% 
loss is a data artifact or a business crisis. This finding should drive our entire EDA and 
modeling strategy.
"""

## 28. created_by
User ID of the person who created the task record.

In [128]:
print(f"Null values: {df['created_by'].isnull().sum()}")
print(f"Null %: {df['created_by'].isnull().mean()*100:.2f}%")
print(f"\nUnique users: {df['created_by'].nunique()}")
print(f"\nTop 5 most active users:")
print(df['created_by'].value_counts())
print(f"\nData type: {df['created_by'].dtype}")


Null values: 0
Null %: 0.00%

Unique users: 119

Top 5 most active users:
created_by
user_026    44
user_087    40
user_028    38
user_070    37
user_012    37
            ..
user_084    18
user_118    18
user_093    17
user_027    15
user_088    15
Name: count, Length: 119, dtype: int64

Data type: object


**Result:** created_by has 0 nulls. 119 unique users. Clean metadata column.

**Observations:**
- All 3,200 tasks have a creator assigned — good data governance.
- Distribution is relatively balanced: top user has only 44 tasks (1.4%), suggesting work is 
  spread across the team rather than concentrated on a few people.
- Format is consistent: user_XXX pattern.

**Decision:** Clean. No action needed. This is administrative metadata, not a business variable. 
Will not be used in modeling unless we discover that certain users systematically create more 
profitable/unprofitable tasks (unlikely, since created_by ≠ executed_by).


## 29. updated_at
Timestamp of the last update to the task record.

In [129]:
print(f"Null values: {df['updated_at'].isnull().sum()}")
print(f"Null %: {df['updated_at'].isnull().mean()*100:.2f}%")
print(f"\nData type: {df['updated_at'].dtype}")
print(f"\nSample values (first 5):")
print(df['updated_at'].head())

# Parse to datetime if not already
if df['updated_at'].dtype == 'object':
    df['updated_at'] = pd.to_datetime(df['updated_at'])
    print(f"\n✓ Parsed to datetime")

print(f"\nDate range:")
print(f"  Earliest: {df['updated_at'].min()}")
print(f"  Latest: {df['updated_at'].max()}")
print(f"  Span: {(df['updated_at'].max() - df['updated_at'].min()).days} days")

Null values: 0
Null %: 0.00%

Data type: object

Sample values (first 5):
0    2025-11-28
1    2026-01-26
2    2025-09-17
3    2025-11-12
4    2026-05-09
Name: updated_at, dtype: object

✓ Parsed to datetime

Date range:
  Earliest: 2025-07-01 00:00:00
  Latest: 2026-06-03 00:00:00
  Span: 337 days



**Result:** updated_at has 0 nulls. Successfully parsed to datetime. 
Range: July 1, 2025 to June 3, 2026 (337 days).

**Observations:**
- Clean metadata column. All records have an update timestamp.
- Date range aligns with created_at (July 2025 - May 2026 per handoff). The updated_at 
  extends slightly beyond (June 3) because tasks created in May were updated later.
- This column was already used in Phase 1 to resolve the 48 duplicate task_ids — we kept 
  the row with the most recent updated_at.

**Decision:** Clean. No action needed. Administrative metadata, not a business variable.


## 30. task_status
Current status of the task in the workflow.

In [130]:
print(f"Null values: {df['task_status'].isnull().sum()}")
print(f"Null %: {df['task_status'].isnull().mean()*100:.2f}%")
print(f"\nUnique statuses: {df['task_status'].nunique()}")
print(f"\nStatus distribution:")
print(df['task_status'].value_counts())

# Check consistency with delivered_at
print(f"\n\nConsistency check with delivered_at:")
print(f"Status = 'delivered' but delivered_at is NULL:")
delivered_no_date = df[(df['task_status'] == 'delivered') & (df['delivered_at'].isnull())]
print(f"  {len(delivered_no_date)} rows (data entry error mentioned in handoff)")

print(f"\nStatus ≠ 'delivered' but delivered_at is NOT NULL:")
not_delivered_has_date = df[(df['task_status'] != 'delivered') & (df['delivered_at'].notnull())]
print(f"  {len(not_delivered_has_date)} rows")

Null values: 0
Null %: 0.00%

Unique statuses: 6

Status distribution:
task_status
in_progress    1109
review          855
delivered       758
draft           179
archived        150
blocked         149
Name: count, dtype: int64


Consistency check with delivered_at:
Status = 'delivered' but delivered_at is NULL:
  13 rows (data entry error mentioned in handoff)

Status ≠ 'delivered' but delivered_at is NOT NULL:
  2417 rows


**Result:** task_status has 0 nulls. 6 unique statuses. Clean categorical column.

**Status distribution:**
- in_progress: 1,109 (35%) — tasks currently being worked on
- review: 855 (27%) — tasks under review
- delivered: 758 (24%) — completed and delivered
- draft: 179 (6%) — not yet started
- archived: 150 (5%) — completed and archived
- blocked: 149 (5%) — stuck/waiting

**Consistency with delivered_at:**
- **13 rows:** status = 'delivered' but delivered_at is NULL — data entry error confirmed 
  from handoff. These tasks are marked delivered but missing the delivery timestamp.
- **2,417 rows:** status ≠ 'delivered' but delivered_at is NOT NULL — this is expected! 
  These are tasks that WERE delivered but have moved to other workflow stages (review, 
  archived). In a real workflow, tasks don't stay in 'delivered' status forever.

**Key insight:** Only 758 tasks (24%) are currently in 'delivered' status, but 3,162 tasks 
(99%) have a delivered_at date. This means most tasks flow through: delivered → review → 
archived. The workflow is dynamic.

**Decision:** Clean. The 13 delivered-but-no-date tasks are noted but left as-is (consistent 
with handoff approach). For analysis, we should filter on delivered_at NOT NULL, not on 
status = 'delivered', to capture all completed work.


## 31. workflow_stage
Current stage in the task workflow.

In [131]:
print(f"Null values: {df['workflow_stage'].isnull().sum()}")
print(f"Null %: {df['workflow_stage'].isnull().mean()*100:.2f}%")
print(f"\nUnique stages: {df['workflow_stage'].nunique()}")
print(f"\nStage distribution:")
print(df['workflow_stage'].value_counts())

Null values: 0
Null %: 0.00%

Unique stages: 5

Stage distribution:
workflow_stage
qa               683
client_review    651
finalized        637
briefing         631
execution        598
Name: count, dtype: int64



**Result:** workflow_stage has 0 nulls. 5 unique stages. Clean categorical column.

**Stage distribution (all relatively balanced):**
- qa: 683 (21%) — quality assurance phase
- client_review: 651 (20%) — under client review
- finalized: 637 (20%) — completed and finalized
- briefing: 631 (20%) — initial briefing/planning
- execution: 598 (19%) — active execution

**Observations:**
- Distribution is nearly uniform (19-21% each) — suggests tasks are well-distributed across 
  the workflow pipeline, not bottlenecked at any stage.
- This is different from task_status (which had in_progress at 35%). workflow_stage appears 
  to track the sequential process flow, while task_status tracks operational state.

**Decision:** Clean. No action needed. This is workflow metadata. 


## 32. jira_ticket
JIRA ticket ID associated with the task.

In [132]:
print(f"Null values: {df['jira_ticket'].isnull().sum()}")
print(f"Null %: {df['jira_ticket'].isnull().mean()*100:.2f}%")
print(f"\nUnique tickets: {df['jira_ticket'].nunique()}")
print(f"\nData type: {df['jira_ticket'].dtype}")
print(f"\nSample values (first 10 non-null):")
print(df[df['jira_ticket'].notnull()]['jira_ticket'].head(10).tolist())

# Check format consistency
if df['jira_ticket'].notnull().any():
    sample = df[df['jira_ticket'].notnull()]['jira_ticket'].iloc[0]
    print(f"\nFormat example: {sample}")
    # Check if all follow JIRA-XXXXX pattern
    non_null = df[df['jira_ticket'].notnull()]['jira_ticket']
    has_jira_prefix = non_null.str.startswith('JIRA-').sum()
    print(f"Tickets with 'JIRA-' prefix: {has_jira_prefix}/{len(non_null)}")

Null values: 332
Null %: 10.38%

Unique tickets: 2828

Data type: object

Sample values (first 10 non-null):
['JIRA-49014', 'JIRA-84793', 'JIRA-42485', 'JIRA-53111', 'JIRA-86006', 'JIRA-60375', 'JIRA-11083', 'JIRA-78764', 'JIRA-77681', 'JIRA-82158']

Format example: JIRA-49014
Tickets with 'JIRA-' prefix: 2868/2868


**Result:** jira_ticket has 332 nulls (10.4%). 2,868 non-null values with 2,828 unique tickets.

**Observations:**
- Format is consistent: all follow JIRA-XXXXX pattern (100% compliance).
- **Nearly 1:1 mapping:** 2,828 unique tickets / 2,868 non-null = 98.6% unique. Most tasks 
  have their own JIRA ticket, with only ~40 tickets shared across multiple tasks.
- 10.4% of tasks have no JIRA ticket — could be: (a) small/internal tasks not tracked in JIRA, 
  (b) pre-JIRA adoption tasks, or (c) missing data entry.

**Decision:** Leave as-is. The nulls are documented (handoff mentioned 339, we found 332 after 
duplicate removal). This is administrative metadata. Could be useful in EDA to check if tasks 
WITHOUT JIRA tickets have different profit patterns (e.g., less formal tracking = worse outcomes).


## 33. legacy_ai_flag
Flag indicating if the task used legacy AI tools (vs modern AI).

In [133]:
print(f"Null values: {df['legacy_ai_flag'].isnull().sum()}")
print(f"Null %: {df['legacy_ai_flag'].isnull().mean()*100:.2f}%")
print(f"\nUnique values: {df['legacy_ai_flag'].nunique()}")
print(f"\nValue distribution:")
print(df['legacy_ai_flag'].value_counts())
print(f"\nData type: {df['legacy_ai_flag'].dtype}")

# Check relationship with ai_assisted
print(f"\n\nCross-tabulation with ai_assisted:")
print(pd.crosstab(df['legacy_ai_flag'], df['ai_assisted'], margins=True))

# AI usage % for legacy vs non-legacy
print(f"\n\nAI usage by legacy flag:")
print(df.groupby('legacy_ai_flag')['ai_usage_pct'].agg(['count', 'mean', 'median']))

Null values: 0
Null %: 0.00%

Unique values: 3

Value distribution:
legacy_ai_flag
false      1436
true       1428
unknown     336
Name: count, dtype: int64

Data type: object


Cross-tabulation with ai_assisted:
ai_assisted     False  True   All
legacy_ai_flag                   
false             329  1107  1436
true              284  1144  1428
unknown            64   272   336
All               677  2523  3200


AI usage by legacy flag:
                count  mean  median
legacy_ai_flag                     
false            1377  0.36    0.35
true             1359  0.37    0.34
unknown           321  0.36    0.35


**Result:** legacy_ai_flag has 0 nulls. 3 values: false (1,436), true (1,428), unknown (336).

**Observations:**
- Nearly **50/50 split** between legacy (1,428) and modern (1,436) AI tools — Alkemy is in 
  transition between old and new AI systems.
- 10.5% unknown — likely tasks where AI tool type wasn't tracked.
- **No difference in AI usage %:** legacy=37%, modern=36%, unknown=36%. The TYPE of AI tool 
  doesn't affect HOW MUCH it's used.

**Cross-tab with ai_assisted:**
- legacy=true: 80% AI-assisted (1,144/1,428)
- legacy=false: 77% AI-assisted (1,107/1,436)
- unknown: 81% AI-assisted (272/336)
All three groups have similar AI adoption rates.

**Key insight:** Legacy vs modern AI tools show NO difference in usage intensity. This suggests 
the transition to modern AI hasn't changed behavior — teams use both tools similarly. However, 
we should check in EDA if legacy vs modern AI affects OUTCOMES (profit, rework, quality) even 
if usage % is the same.

**Decision:** Clean. Leave as-is. This could be an important segmentation variable: does legacy 
AI have different rework patterns or profit impact than modern AI, even at same usage levels?


## 34. content_version
Version of the content/deliverable.

In [134]:
print(f"Null values: {df['content_version'].isnull().sum()}")
print(f"Null %: {df['content_version'].isnull().mean()*100:.2f}%")
print(f"\nUnique versions: {df['content_version'].nunique()}")
print(f"\nVersion distribution:")
print(df['content_version'].value_counts().sort_index())
print(f"\nData type: {df['content_version'].dtype}")

# Relationship with revisions
print(f"\n\nRelationship with revisions:")
print(df.groupby('content_version')['revisions'].agg(['count', 'mean', 'median']))

# Relationship with rework_hours
print(f"\n\nRelationship with rework_hours:")
print(df.groupby('content_version')['rework_hours'].agg(['count', 'mean', 'median']))

Null values: 0
Null %: 0.00%

Unique versions: 5

Version distribution:
content_version
final    552
v1       910
v2       769
v3       655
v4       314
Name: count, dtype: int64

Data type: object


Relationship with revisions:
                 count  mean  median
content_version                     
final              552  2.98    3.00
v1                 910  3.10    3.00
v2                 769  3.05    3.00
v3                 655  2.98    3.00
v4                 314  2.80    3.00


Relationship with rework_hours:
                 count  mean  median
content_version                     
final              542  2.61    1.83
v1                 889  2.34    1.83
v2                 749  2.40    1.84
v3                 638  2.47    1.80
v4                 311  2.42    1.63



**Result:** content_version has 0 nulls. 5 unique versions: v1 (910), v2 (769), v3 (655), 
v4 (314), final (552).

**Observations:**
- Distribution: v1 is most common (28%), followed by v2 (24%), v3 (20%), final (17%), v4 (10%).
- **Surprising: no correlation with revisions or rework.** All versions have nearly identical 
  revision counts (mean ~3, median 3) and rework hours (mean ~2.4h, median ~1.8h).
- This suggests content_version does NOT mean "number of iterations" but rather refers to the 
  VERSION OF THE TEMPLATE/FORMAT used (e.g., v1 of blog post template, v2 of blog post template).

**Hypothesis:** content_version likely tracks Alkemy's content standards evolution (e.g., brand 
guidelines v1 vs v2 vs final version). It's metadata about the deliverable format, not the task 
iteration count.

**Decision:** Clean. Leave as-is. May be useful in EDA to check if certain content versions 
correlate with better/worse outcomes, but unlikely to be a key variable for the AI threshold 
analysis.
